# When is a detour worth it?

Every place has a cost per unit distance. A route's total cost adds up **local cost × distance traveled**:

$$J[q]=\int_0^1 c(q)\|q'\|\,dt,\qquad c(q)=1+\sum_i w_i e^{-\|q-o_i\|^2/\sigma_i^2}.$$

Traveling the same curve faster does not change this quantity. Hills are expensive, not forbidden.
This notebook uses the same Python solver as the website to find and compare candidate routes.


In [ ]:
%matplotlib inline
import json
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from path_planning_ode import Scene, cost_field, presets, solve, weighted_distance

scene = presets()["asymmetric"]
result = solve(scene)
names = {"straight": "Direct", "bend-x": "Right arc", "bend-y": "Left arc"}
direct_cost = weighted_distance(np.array([scene.start, scene.end]), scene)
print(f"Direct endpoint-to-endpoint route: cost {direct_cost:.2f}")
for name, state in result.final.items():
    print(f"{names[name]:10} cost={state.cost:.2f}, distance={state.length:.2f}, {state.status}")

## Which side is cheaper?

Try one direct starting route and one arc on either side. The labels identify their starting shapes, not a constraint on the final route. Compare total cost and distance separately. A longer route can be cheaper; a converged route need not be a minimum.


In [ ]:
x, y = np.meshgrid(np.linspace(-4, 14, 160), np.linspace(-4, 14, 160))
c = cost_field(np.stack([x, y], axis=-1), scene.obstacles)
fig, ax = plt.subplots(figsize=(7, 7))
heatmap = ax.contourf(x, y, c, levels=18, cmap="YlOrBr", alpha=0.5)
fig.colorbar(heatmap, ax=ax, label="Cost per unit distance")
for name, state in result.final.items():
    ax.plot(*state.path.T, label=f"{names[name]}: cost {state.cost:.2f}")
ax.scatter(*np.array([scene.start, scene.end]).T, c="black")
ax.set_aspect("equal")
ax.legend()
ax.set_title("One landscape, different route costs")
plt.show()

## Strength versus width

Strength changes the price of crossing a hill. Width changes both the extent of the expensive region and the distance needed to avoid it. Vary one at a time. These are the costs of the three final candidate polylines, not a certified global minimum.


In [ ]:
central = presets()["central"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, (parameter, values) in zip(
    axes,
    [
        ("weight", [0, 0.5, 1, 2, 4, 6, 10]),
        ("width", [0.5, 1, 1.5, 2, 2.5, 3]),
    ],
):
    costs = {name: [] for name in central.guesses}
    for value in values:
        bump = replace(central.obstacles[0], **{parameter: value})
        experiment = replace(central, obstacles=(bump,))
        candidates = solve(experiment).final
        for name, state in candidates.items():
            costs[name].append(state.cost)
            if state.status != "converged":
                print(parameter, value, names[name], state.status)
    for name, values_cost in costs.items():
        axis.plot(values, values_cost, marker="o", label=names[name])
    axis.set_xlabel("Strength" if parameter == "weight" else "Width")
    axis.set_ylabel("Final candidate cost")
    axis.legend()
plt.tight_layout()
plt.show()

## The same route has the same cost

Subdividing straight segments changes the sampling, not the route. The diagnostic integrates each Gaussian along the complete segment, so it also counts narrow hills lying between vertices.


In [ ]:
coarse = np.array([scene.start, scene.end])
nonuniform = coarse[0] + np.array([0, 0.01, 0.2, 0.9, 1])[:, None] * (coarse[1] - coarse[0])
assert np.isclose(weighted_distance(coarse, scene), weighted_distance(nonuniform, scene))
assert np.isclose(weighted_distance(coarse, scene), weighted_distance(coarse[::-1], scene))
print("Subdivision and reversal preserve route cost:", weighted_distance(coarse, scene))

## Optional: what the solver is doing

To fix the free traversal schedule, we use energy with integrand c² times squared parameter speed. For each geometric route, its minimum over parameterizations is weighted distance squared. Euler–Lagrange supplies the ODE; finite differences and damped Newton find stationary candidates.

Newton reduces **ODE residual**, not necessarily total cost. A small residual is not proof of minimality or accurate resolution of a narrow hill.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, history in result.histories.items():
    iterations = [state.iteration for state in history]
    axes[0].plot(iterations, [state.cost for state in history], label=names[name])
    axes[1].semilogy(
        iterations, [max(state.residual_norm, 1e-12) for state in history], label=names[name]
    )
axes[0].set_ylabel("Total route cost")
axes[1].set_ylabel("RMS ODE residual")
for ax in axes:
    ax.set_xlabel("Newton iteration")
    ax.legend()
plt.tight_layout()
plt.show()

## Reproduce a browser scene

`Scene.from_dict` accepts the website's JSON export. To read a downloaded scene, use `json.loads(Path("path-scene.json").read_text())` after importing `Path` from `pathlib`.


In [ ]:
restored = Scene.from_dict(json.loads(json.dumps(scene.to_dict())))
reproduced = solve(restored)
for name in result.final:
    np.testing.assert_allclose(result.final[name].path, reproduced.final[name].path)
    assert np.isclose(result.final[name].cost, reproduced.final[name].cost)
print("Scene round trip reproduced all paths and costs.")

## Keep exploring

- Move a hill and watch which side becomes cheaper.
- Compare 15, 31, and 63 interior points. Do geometry and cost stabilize?
- Try a zero-strength hill or coincident endpoints.
- Inspect non-converged candidates before trusting their shape.

Read the [full derivation](https://twallengren.github.io/path-planning-ode/derivation.html) for the objective, ODE, Jacobian, and stopping rules.
